# Batch 평가 입력

Python 3.12 새 커널에서 세 사례를 Batch JSONL로 변환부터 확인합니다.

## 개념을 코드에 연결하기

**호출 표기와 실행 시점.** JSONL은 한 줄에 독립된 JSON 객체 하나를 저장하는 형식입니다. 각 요청은 custom_id, HTTP method, url, body를 가지며 custom_id가 제출 결과를 원래 평가 사례와 다시 연결합니다. 제출 전에 중복 ID, 필수 필드, 허용 모델을 로컬에서 검사합니다.

**다른 상황에 적용.** 천 개의 답변을 평가할 때 사례 ID를 judge-Q002처럼 안정적으로 만들면 결과 도착 순서가 달라도 원본 질문과 결합할 수 있습니다. 실패 행만 다시 제출할 때도 같은 ID 정책이 추적에 도움이 됩니다.

**비교와 경계.** 파일 형식 검증은 원격 제출 성공이나 평가 품질을 보장하지 않습니다. 이 절은 Batch 입력을 만들고 모의 결과를 결합하는 데까지 다루며 실제 API 제출은 실행하지 않습니다. 원격 제한과 모델 지원 여부는 제출 시점 공식 문서를 다시 확인해야 합니다.


## Python 3.12 패키지 버전 고정

**이 셀의 코드 읽기.** `%pip install`은 이 노트북 커널이 사용할 패키지를 설치합니다. 앞 셀의 값을 입력으로 사용하지 않으며, 설치가 끝난 뒤의 코드 셀에서 패키지를 가져와 사용합니다.


In [1]:
# 실습 환경에서 사용하는 보조 패키지 버전을 고정합니다.
%pip install -q python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## 세 사례를 Batch JSONL로 변환

**이 셀의 코드 읽기.** `json.loads(text)`는 JSON 문자열을 파이썬 딕셔너리나 목록으로 변환해 반환합니다. `read_text()`는 파일 내용을 문자열로 반환합니다. `write_text(text)`가 실행되는 줄에서 문자열이 실제 파일에 저장됩니다. `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [2]:
# 세 평가 사례를 Responses endpoint용 Batch JSONL 행으로 변환해 저장합니다.
from pathlib import Path
import json
root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'labs/part4/ch15-evaluation-advanced').is_dir())
data = json.loads((root / 'labs/part4/ch15-evaluation-advanced/data/evaluation-cases.json').read_text())
out = root / 'labs/part4/ch15-evaluation-advanced/runs/batch-eval-input.jsonl'
rows = []
for case in data['cases']:
    rows.append({'custom_id': f"judge-{case['qid']}", 'method': 'POST',
      'url': '/v1/responses', 'body': {'model': 'gpt-5.4-mini',
      'input': f"질문: {case['question']}\n응답: {case['response']}"}})
out.parent.mkdir(exist_ok=True)
out.write_text(''.join(json.dumps(x, ensure_ascii=False) + '\n' for x in rows))
print('저장:', out.name, '행:', len(rows))

저장: batch-eval-input.jsonl 행: 3


## 요청 형식과 요청 식별자 검증

**이 셀의 코드 읽기.** `json.loads(text)`는 JSON 문자열을 파이썬 딕셔너리나 목록으로 변환해 반환합니다. `read_text()`는 파일 내용을 문자열로 반환합니다. `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [3]:
# 각 JSONL 행의 식별자·method·endpoint·body 필드를 제출 전에 검사합니다.
loaded = [json.loads(line) for line in out.read_text().splitlines()]
ids = [row['custom_id'] for row in loaded]
checks = {
 'json_rows': len(loaded) == len(rows),
 'unique_custom_id': len(ids) == len(set(ids)),
 'post_only': all(x['method'] == 'POST' for x in loaded),
 'responses_endpoint': all(x['url'] == '/v1/responses' for x in loaded),
 'body_fields': all(x['body'].get('model') and x['body'].get('input') for x in loaded),
}
print(checks)
assert all(checks.values())

{'json_rows': True, 'unique_custom_id': True, 'post_only': True, 'responses_endpoint': True, 'body_fields': True}


## 모의 결과의 누락·중복·알 수 없는 ID 검사

**이 셀의 코드 읽기.** `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [4]:
# 로컬에서 실행한 범위와 실제 업로드·완료·청구 미확인 범위를 구분합니다.
mock_results = [
 {'custom_id': ids[0]}, {'custom_id': ids[1]},
 {'custom_id': ids[1]}, {'custom_id': 'judge-UNKNOWN'},
]
result_ids = [row['custom_id'] for row in mock_results]
missing = sorted(set(ids) - set(result_ids))
unknown = sorted(set(result_ids) - set(ids))
duplicates = sorted({value for value in result_ids if result_ids.count(value) > 1})
print({'누락': missing, '알 수 없는 ID': unknown, '중복': duplicates})
print('실제 업로드·완료·할인 청구: 미확인')
assert missing and unknown and duplicates

{'누락': ['judge-Q090'], '알 수 없는 ID': ['judge-UNKNOWN'], '중복': ['judge-Q006']}
실제 업로드·완료·할인 청구: 미확인
